# Lenormand B4-P — Anchor–Verifier–Calibrator (Factor Track)

**目标不是换一个更大的分类 backbone，而是把 24-label list generation 变成 24 个共享参数的语义核验任务。**

本 notebook 运行：

1. 冻结的 24 张 Label Card；
2. 用户与重复文本隔离的 3-fold OOF；
3. label-conditioned 长帖 clause 视图；
4. 真实正例 + 语义近邻 hard negative 检索；
5. **一个**共享 `Qwen3-14B` NF4 QLoRA verifier；
6. 全量 `Qwen3-30B-A3B-Instruct-2507` frozen anchor；
7. 逐标签非负 OOF 校准与 support-aware threshold；
8. 三折 verifier 测试集成，并可把 factors 合并回现有 `Lenormand.csv`。

默认先跑 Fold 0。它会缓存每个 chunk，因此断线后重新执行同一 cell 会续跑。只有 Stage-A 门控通过后，才打开三折和测试推理。所有原始比赛帖子只在本地 Colab/Drive 流转，不调用外部 API。


In [ ]:
#@title 0. 安装依赖（Colab A100）
%%capture
!pip install -q -U   "transformers>=4.51.0"   "accelerate>=1.5.0"   "peft>=0.15.0"   "bitsandbytes>=0.45.0"   "sentencepiece>=0.2.0"   "openpyxl>=3.1.0"   "scikit-learn>=1.5.0"   "scipy>=1.13.0"


In [ ]:
#@title 1. Drive、路径和运行开关
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess, sys, time, json

ROOT = Path('/content/drive/MyDrive/IEEE_BigData2026')
TRAIN_PATH = ROOT / 'train.xlsx'
TEST_PATH = ROOT / 'leaderboard.xlsx'
ARTIFACT_ROOT = ROOT / 'results' / 'B4P_AVC_FAST3'
MODULE_PATH = ROOT / 'b4p_anchor_verifier.py'
B1_PATH = ROOT / 'b1_experiments.py'
BASELINE_SUBMISSION = ROOT / 'Lenormand.csv'  # 用现有 risk/evidence，仅替换 factors

# 第一次运行建议保持如下：先做 benchmark + Fold 0。
RUN_BENCHMARK = True
RUN_STAGE_A_FOLD0 = True
RUN_FULL_OOF = False       # Stage-A 通过后改 True
RUN_TEST = False           # OOF >= 0.60 后改 True
FORCE_TEST_BELOW_GATE = False
OVERWRITE_ADAPTERS = False

# 可选：把 B3/ModernBERT 的严格 OOF/test NPZ 接入非负 stack。
# NPZ 必须有 row_ids，以及 logits 或 probabilities；键名在 OOF/TEST 两边一致。
OPTIONAL_OOF_COMPONENTS = {
    # 'B3_RESIDUAL': ROOT / 'results/.../B3_oof.npz',
}
OPTIONAL_TEST_COMPONENTS = {
    # 'B3_RESIDUAL': ROOT / 'results/.../B3_test.npz',
}

ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

missing = [path for path in (MODULE_PATH, B1_PATH) if not path.exists()]
if missing:
    print('请上传缺失模块：', [p.name for p in missing])
    uploaded = files.upload()
    for path in missing:
        if path.name not in uploaded:
            raise FileNotFoundError(path)
        shutil.copy2('/content/' + path.name, path)

assert TRAIN_PATH.exists(), TRAIN_PATH
assert TEST_PATH.exists(), TEST_PATH

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout)
print('Artifacts:', ARTIFACT_ROOT)


In [ ]:
#@title 2. 导入 B4-P 并冻结配置
sys.path.insert(0, str(ROOT))
import importlib
import numpy as np
import pandas as pd
import torch
import b1_experiments as b1
import b4p_anchor_verifier as b4
importlib.reload(b1)
importlib.reload(b4)

CFG = b4.B4PConfig(
    seed=42,
    n_splits=3,
    verifier_model='Qwen/Qwen3-14B',
    anchor_model='Qwen/Qwen3-30B-A3B-Instruct-2507',
    retriever_model='intfloat/e5-large-v2',
    max_length=2048,
    include_retrieval=True,
    sft_epochs=1.0,
    sft_batch_size=1,
    sft_gradient_accumulation=32,
    verifier_score_batch_size=8,
    anchor_score_batch_size=12,
)

b4.seed_everything(CFG.seed)
torch.set_float32_matmul_precision('high')
print(json.dumps(b4.asdict(CFG), indent=2))


In [ ]:
#@title 3. 读取数据并建立严格 grouped folds
bundle = b1.load_training_data(ROOT, TRAIN_PATH)
train_corpus = b4.training_corpus(bundle)
test_corpus = b4.load_test_data(ROOT, TEST_PATH)

fold_path = ARTIFACT_ROOT / f'folds_{CFG.n_splits}_seed{CFG.seed}.npz'
if fold_path.exists():
    saved = np.load(fold_path, allow_pickle=True)
    assert saved['row_ids'].astype(str).tolist() == bundle.row_ids.astype(str).tolist()
    folds = saved['folds'].astype(int)
else:
    folds = b1.make_leak_safe_folds(bundle, n_splits=CFG.n_splits, seed=CFG.seed)
    np.savez_compressed(fold_path, row_ids=bundle.row_ids, folds=folds)

fold_report = b4.validate_folds(bundle, folds)
card_hash = b4.freeze_cards(ARTIFACT_ROOT / 'FROZEN_LABEL_CARDS.json')
print('train/test:', len(bundle.texts), len(test_corpus.texts))
print('factor supports:', dict(zip(b4.FACTOR_LABELS, bundle.factor_binary.sum(axis=0).tolist())))
print('fold report:', fold_report)
print('frozen card sha256:', card_hash)


### 为什么先建语义缓存？

检索库本身只有 1,635 条，没必要反复重建 FAISS。这里一次性编码 document/clause/card，之后每折仅通过布尔 mask 控制可检索范围。长帖不是粗暴保留开头，而是对每个 factor 选择相关 clause、首尾语境并保持原文顺序。


In [ ]:
#@title 4. 一次性构建 train semantic cache（约数分钟）
train_cache = b4.prepare_semantic_cache(
    train_corpus,
    CFG,
    ARTIFACT_ROOT / 'semantic_cache' / 'train',
)
print(train_cache.doc_embeddings.shape, train_cache.clause_embeddings.shape, train_cache.card_embeddings.shape)


In [ ]:
#@title 5. Prompt/provenance smoke test
fold0_train = np.flatnonzero(folds != 0)
fold0_valid = np.flatnonzero(folds == 0)
retriever = b4.FoldRetriever(bundle, train_corpus, train_cache)
sample_prompts = b4.build_prompt_table(
    train_corpus,
    train_cache,
    train_corpus,
    retriever,
    fold0_train,
    CFG,
    query_rows=fold0_valid[:8],
    train_targets=bundle.factor_binary,
    query_is_training_corpus=True,
)
print('sample pairs:', len(sample_prompts))
print(sample_prompts[['pair_id', 'label', 'positive_index', 'negative_index', 'target']].head())
print('\n--- one actual prompt ---\n')
print(sample_prompts.iloc[0].prompt[:6000])


In [ ]:
#@title 6. 192-pair 真吞吐 benchmark（运行前不要猜时间）
if RUN_BENCHMARK:
    benchmark = b4.benchmark_models(
        sample_prompts.prompt.tolist(),
        CFG,
        ARTIFACT_ROOT / 'BENCHMARK',
        run_anchor=True,
        run_verifier_base=True,
    )
    display(benchmark)
    print('说明：projected_13080_pair_minutes 是一折验证集的单模型纯推理时间，不含 14B 训练。')
else:
    print('RUN_BENCHMARK=False，已跳过。')


## Stage A — 只跑 Fold 0

这一段会训练一个共享 14B adapter，然后让 14B Verifier 与 30B Anchor 分别对 Fold 0 的所有 `验证帖子 × 24 factors` 打 A/B margin。Stage-A 的主门控是 **Macro-AP**（阈值无关）；`A/B zero-margin F1` 只检查模型有没有坍塌，不能当正式 OOF F1。


In [ ]:
#@title 7. 可续跑的单折函数
def run_outer_fold(fold: int):
    fold_dir = ARTIFACT_ROOT / 'OOF' / f'fold_{fold}'
    fold_dir.mkdir(parents=True, exist_ok=True)
    result_path = fold_dir / 'fold_logits.npz'
    valid = np.flatnonzero(folds == fold)
    train = np.flatnonzero(folds != fold)

    if result_path.exists():
        saved = np.load(result_path, allow_pickle=True)
        if saved['row_ids'].astype(str).tolist() == bundle.row_ids.astype(str).tolist():
            print('[resume]', result_path)
            return saved['verifier_logits'], saved['anchor_logits']

    started = time.perf_counter()
    adapter = b4.train_one_outer_fold(
        bundle, folds, train_cache, fold, CFG,
        ARTIFACT_ROOT / 'OOF',
        overwrite=OVERWRITE_ADAPTERS,
    )
    print('adapter:', adapter)

    verifier_logits, _ = b4.run_verifier_scoring(
        adapter,
        train_corpus, train_cache,
        train_corpus, bundle, train_cache,
        train, CFG,
        fold_dir / 'verifier_scores',
        query_rows=valid,
        query_is_training_corpus=True,
    )
    np.savez_compressed(
        fold_dir / 'verifier_only.npz', row_ids=bundle.row_ids,
        logits=verifier_logits, valid_indices=valid,
    )

    anchor_logits, _ = b4.run_anchor_scoring(
        train_corpus, train_cache,
        train_corpus, bundle, train_cache,
        train, CFG,
        fold_dir / 'anchor_scores',
        query_rows=valid,
        query_is_training_corpus=True,
    )
    np.savez_compressed(
        result_path,
        row_ids=bundle.row_ids,
        valid_indices=valid,
        verifier_logits=verifier_logits,
        anchor_logits=anchor_logits,
        elapsed_minutes=(time.perf_counter() - started) / 60,
    )
    return verifier_logits, anchor_logits


In [ ]:
#@title 8. 开跑 Stage-A Fold 0
stage_a = None
if RUN_STAGE_A_FOLD0:
    v0, a0 = run_outer_fold(0)
    valid = folds == 0
    stage_a = pd.DataFrame([
        {'model': 'B4P_SHARED_SFT', **b4.diagnostic_fold_metrics(v0[valid], bundle.factor_binary[valid])},
        {'model': 'B4P_FULL_ANCHOR', **b4.diagnostic_fold_metrics(a0[valid], bundle.factor_binary[valid])},
    ]).sort_values('macro_ap', ascending=False)
    display(stage_a)
    stage_a_gate = bool(stage_a.macro_ap.max() >= 0.48 and stage_a.macro_ap.min() >= 0.40)
    print({'stage_a_gate': stage_a_gate, 'rule': 'best AP >= .48 and both AP >= .40'})
    print('若 gate=True：把 RUN_FULL_OOF 改成 True，继续第 9 格。若明显失败，先不要烧另外两折。')
else:
    print('RUN_STAGE_A_FOLD0=False，已跳过。')


## Full OOF — 正式决定 B4-P 是否成立

这里的 F1 才是严格结果：每个 outer-valid 行的 Anchor/Verifier 都没见过该用户、重复文本或标签；stack 权重和 thresholds 也只从其他 OOF 行学习。核心接受线为 Macro-F1 ≥ 0.60，冲榜期望是接近 0.67。


In [ ]:
#@title 9. 跑完三折并做非负 OOF stack
decision = None
components_oof = None
if RUN_FULL_OOF:
    verifier_oof = np.full(bundle.factor_binary.shape, np.nan, dtype=np.float32)
    anchor_oof = np.full(bundle.factor_binary.shape, np.nan, dtype=np.float32)
    for fold in range(CFG.n_splits):
        verifier_fold, anchor_fold = run_outer_fold(fold)
        valid = folds == fold
        verifier_oof[valid] = verifier_fold[valid]
        anchor_oof[valid] = anchor_fold[valid]

    components_oof = {
        'B4P_SHARED_SFT': verifier_oof,
        'B4P_FULL_ANCHOR': anchor_oof,
    }
    for name, path in OPTIONAL_OOF_COMPONENTS.items():
        components_oof[name] = b4.load_aligned_component(path, bundle.row_ids)

    np.savez_compressed(
        ARTIFACT_ROOT / 'B4P_CORE_OOF.npz',
        row_ids=bundle.row_ids, folds=folds, targets=bundle.factor_binary,
        verifier_logits=verifier_oof, anchor_logits=anchor_oof,
    )
    decision = b4.evaluate_b4p_oof(
        components_oof, bundle, folds, CFG,
        ARTIFACT_ROOT / 'OOF_EVALUATION',
    )
    display(pd.read_csv(ARTIFACT_ROOT / 'OOF_EVALUATION' / 'B4P_OOF_SUMMARY.csv'))
    print(json.dumps(decision, indent=2))
else:
    print('RUN_FULL_OOF=False。Stage-A 通过后开启。')


## Test inference 与提交合并

Verifier 使用三折 adapter 对 378 条测试集分别预测后平均；Anchor 用全部训练记忆预测一次。最终 calibrator 只从严格 OOF logits 学习。若 `Lenormand.csv` 存在，将只替换 `factors`，保留现有 risk/evidence，生成新的 `results/B4P_AVC_FAST3/FINAL/Lenormand.csv`。


In [ ]:
#@title 10. 测试推理（只有 OOF 过线后开启）
test_decision = None
if RUN_TEST:
    if components_oof is None:
        core = np.load(ARTIFACT_ROOT / 'B4P_CORE_OOF.npz', allow_pickle=True)
        assert core['row_ids'].astype(str).tolist() == bundle.row_ids.astype(str).tolist()
        components_oof = {
            'B4P_SHARED_SFT': core['verifier_logits'],
            'B4P_FULL_ANCHOR': core['anchor_logits'],
        }
        for name, path in OPTIONAL_OOF_COMPONENTS.items():
            components_oof[name] = b4.load_aligned_component(path, bundle.row_ids)
    if decision is None:
        with open(ARTIFACT_ROOT / 'OOF_EVALUATION' / 'B4P_DECISION.json') as handle:
            decision = json.load(handle)
    if not decision['core_gate']['passed'] and not FORCE_TEST_BELOW_GATE:
        raise RuntimeError('OOF Macro-F1 未到 .60；若仍要推理，请显式设 FORCE_TEST_BELOW_GATE=True')

    test_cache = b4.prepare_semantic_cache(
        test_corpus, CFG, ARTIFACT_ROOT / 'semantic_cache' / 'test'
    )
    verifier_test_folds = []
    for fold in range(CFG.n_splits):
        adapter = ARTIFACT_ROOT / 'OOF' / f'fold_{fold}' / 'verifier' / 'adapter_final'
        if not (adapter / 'adapter_config.json').exists():
            raise FileNotFoundError(adapter)
        allowed = np.flatnonzero(folds != fold)
        fold_logits, _ = b4.run_verifier_scoring(
            adapter,
            test_corpus, test_cache,
            train_corpus, bundle, train_cache,
            allowed, CFG,
            ARTIFACT_ROOT / 'TEST' / f'verifier_fold_{fold}',
        )
        verifier_test_folds.append(fold_logits)
    verifier_test = np.mean(np.stack(verifier_test_folds), axis=0)

    anchor_test, _ = b4.run_anchor_scoring(
        test_corpus, test_cache,
        train_corpus, bundle, train_cache,
        np.arange(len(bundle.texts)), CFG,
        ARTIFACT_ROOT / 'TEST' / 'anchor_full_memory',
    )
    components_test = {
        'B4P_SHARED_SFT': verifier_test,
        'B4P_FULL_ANCHOR': anchor_test,
    }
    for name, path in OPTIONAL_TEST_COMPONENTS.items():
        components_test[name] = b4.load_aligned_component(path, test_corpus.row_ids)

    baseline = BASELINE_SUBMISSION if BASELINE_SUBMISSION.exists() else None
    test_decision = b4.finalize_test_predictions(
        components_oof, components_test,
        bundle, folds, test_corpus, CFG,
        ARTIFACT_ROOT / 'FINAL',
        baseline_submission=baseline,
    )
    print(json.dumps(test_decision, indent=2))
else:
    print('RUN_TEST=False。请先看严格 OOF，不要提前烧测试推理。')


In [ ]:
#@title 11. 最终完整性检查
print('Cards:', len(b4.LABEL_CARDS))
print('Train pairs:', len(bundle.texts) * len(b4.FACTOR_LABELS))
print('Test pairs:', len(test_corpus.texts) * len(b4.FACTOR_LABELS))
print('Artifact root:', ARTIFACT_ROOT)

if (ARTIFACT_ROOT / 'OOF_EVALUATION' / 'B4P_OOF_SUMMARY.csv').exists():
    display(pd.read_csv(ARTIFACT_ROOT / 'OOF_EVALUATION' / 'B4P_OOF_SUMMARY.csv'))
if (ARTIFACT_ROOT / 'FINAL' / 'B4P_factor_predictions.csv').exists():
    pred = pd.read_csv(ARTIFACT_ROOT / 'FINAL' / 'B4P_factor_predictions.csv')
    assert pred.row_id.astype(str).tolist() == test_corpus.row_ids.astype(str).tolist()
    assert len(pred) == 378
    display(pred.head())
    print('Ready:', ARTIFACT_ROOT / 'FINAL')


### 读数规则

- Stage-A 只看 Macro-AP 和是否坍塌，不对同一验证折调阈值后自我庆祝。
- Full OOF 的 `B4P_NONNEGATIVE_STACK` 才可与内部系统比较。
- **≥ 0.60**：B4 的 verifier 机制成立，值得跑测试；**0.55–0.60**：只做 residual expert，不单独提交；**< 0.55**：停止这条昂贵路线。
- 即使 OOF ≥ 0.60，也先做一次只替换 factors 的受控 leaderboard submission，不能用一次线上分数反复手调测试集。
